# 01 — Generowanie świata i wizualizacja 3D

**Cel.** Korzystając z modułów `src.environments.abstraction.*`, wczytujemy plik
konfiguracyjny środowiska (`configs/environment/<name>.yaml`), parametryzujemy
świat (`WorldData`) i generujemy przeszkody (`ObstaclesData`) wybraną strategią
rozmieszczenia. Wynik renderujemy na czytelnym wykresie 3D z poprawnym
skalowaniem osi.

**Co i jak edytować, by sprawdzać inne wyniki?**

* `CONFIG_NAME` — nazwa pliku w `configs/environment/` bez rozszerzenia
  (`empty`, `forest`, `urban`).
* `MASTER_SEED` — deterministyczne ziarno przekazywane przez `SeedRegistry`.
* `STRATEGY_OVERRIDE` — jeśli różne od `None`, nadpisuje `placement_strategy`
  z YAML (`'strategy_random_uniform'`, `'strategy_grid_jitter'`, `'strategy_empty'`).
* `MAX_DISPLAY_ASPECT` — górna proporcja osi w widoku 3D (zob. komentarz w
  funkcji `_compute_box_aspect`).

**Odniesienia literaturowe.**
Sposób rozmieszczania przeszkód oparto na dwóch rodzinach metod
(Lavalle 2006, *Planning Algorithms*, rozdz. 5): (a) próbkowanie
Monte-Carlo z rejection sampling stref bezpiecznych — `strategy_random_uniform`,
(b) regularne siatki z jitter (Cohen et al., 2009, *Mesh Generation Methods*) —
`strategy_grid_jitter`.

In [ ]:
# Helper dodający `..` do sys.path (umożliwia `from src…` w notebookach).
import prepare_notebook  # noqa: F401

In [ ]:
from pathlib import Path
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import yaml
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from src.environments.abstraction.generate_obstacles import (
    generate_obstacles,
    strategy_random_uniform,
    strategy_grid_jitter,
    strategy_empty,
    ObstaclesData,
)
from src.environments.abstraction.generate_world_boundaries import (
    generate_world_boundaries,
    WorldData,
)
from src.environments.obstacles.ObstacleShape import ObstacleShape
from src.utils.SeedRegistry import SeedRegistry

# ---------------------------------------------------------------------------
# PARAMETRY DO EDYCJI
# ---------------------------------------------------------------------------
CONFIG_NAME: str = "forest"           # 'empty' | 'forest' | 'urban'
MASTER_SEED: int = 43                 # zgodny z `configs/config.yaml`
STRATEGY_OVERRIDE: Optional[str] = None  # None lub jedna z nazw strategii
MAX_DISPLAY_ASPECT: float = 4.0       # górny limit proporcji osi 3D

PROJECT_ROOT = Path(prepare_notebook.project_root)
ENV_CONFIG_PATH = PROJECT_ROOT / "configs" / "environment" / f"{CONFIG_NAME}.yaml"
assert ENV_CONFIG_PATH.exists(), f"Brak pliku konfiguracyjnego: {ENV_CONFIG_PATH}"
print(f"Wczytuję: {ENV_CONFIG_PATH}")

In [ ]:
# Wczytanie konfiguracji Hydry (czysty YAML — bez konieczności inicjalizacji Hydry).
with ENV_CONFIG_PATH.open("r", encoding="utf-8") as f:
    env_cfg = yaml.safe_load(f)

params = env_cfg.get("params", {})
start_positions = np.asarray(env_cfg["initial_xyzs"], dtype=np.float64)
target_positions = np.asarray(env_cfg["end_xyzs"], dtype=np.float64)

print(f"Środowisko: {env_cfg['name']}")
print(f"Liczba dronów (params.num_drones): {params['num_drones']}")
print(f"Wymiary korytarza: {params['track_width']} × {params['track_length']} × {params['track_height']} m")

In [ ]:
# Mapa nazwa-strategii → callable. Sprawdzane są tylko strategie zaimplementowane
# w `src/environments/abstraction/generate_obstacles.py`.
STRATEGY_MAP = {
    "strategy_random_uniform": strategy_random_uniform,
    "strategy_grid_jitter": strategy_grid_jitter,
    "strategy_empty": strategy_empty,
}

strategy_name = STRATEGY_OVERRIDE or params.get("placement_strategy", "strategy_empty")
if strategy_name not in STRATEGY_MAP:
    raise ValueError(f"Nieznana strategia: {strategy_name!r}. Dozwolone: {list(STRATEGY_MAP)}")
placement_strategy = STRATEGY_MAP[strategy_name]

shape_type = ObstacleShape[params.get("shape_type", "BOX")]
size_params = {
    "width": float(params.get("obstacle_width", 5.0)),
    "height": float(params.get("obstacle_height", 5.0)),
    "length": float(params.get("obstacle_length", 5.0)),
}

print(f"Strategia: {strategy_name}")
print(f"Typ przeszkody: {shape_type.name}; wymiary (W×L×H): {size_params}")

In [ ]:
# Generowanie świata i przeszkód z deterministycznym ziarnem.
seeds = SeedRegistry(master_seed=MASTER_SEED)
world: WorldData = generate_world_boundaries(
    width=float(params["track_width"]),
    length=float(params["track_length"]),
    height=float(params["track_height"]),
    ground_height=float(params["ground_position"]),
)

obstacles: ObstaclesData = generate_obstacles(
    world=world,
    n_obstacles=int(params.get("obstacles_number", 0)),
    shape_type=shape_type,
    placement_strategy=placement_strategy,
    size_params=size_params,
    start_positions=start_positions,
    target_positions=target_positions,
    safe_radius=float(params.get("safe_radius", 15.0)),
    rng=seeds.rng("environment"),
)

print(f"Wygenerowane przeszkody: {obstacles.count}")
print(f"Wymiary świata: {world.dimensions} m")
print(f"min_bounds: {world.min_bounds}, max_bounds: {world.max_bounds}")

In [ ]:
# ---------------------------------------------------------------------------
# Pomocnicze funkcje rysunkowe.
# ---------------------------------------------------------------------------
def _compute_box_aspect(world: WorldData, max_aspect: float = 4.0) -> tuple[float, float, float]:
    """Wylicz `box_aspect` dla `Axes3D.set_box_aspect`.

    Naturalna proporcja świata (np. forest 60×600×11 m) bywa zbyt skrajna,
    by patrząc na widok 3D zauważyć rozmieszczenie przeszkód. Stosujemy
    miękkie kompresowanie najdłuższej osi (kierunek misji = Y) do progu
    `max_aspect` × najkrótszej osi. Adnotacja o przeskalowaniu trafia do
    tytułu wykresu (patrz wywołanie poniżej).

    Args:
        world: `WorldData` z wymiarami osi w metrach.
        max_aspect: górny limit stosunku najdłuższej do najkrótszej osi.

    Returns:
        Krotka `(ax_x, ax_y, ax_z)` przekazywana do `set_box_aspect`.
    """
    dims = world.dimensions.astype(float)
    short_axis = float(np.min(dims))
    capped = np.minimum(dims, short_axis * max_aspect)
    return tuple(float(v) for v in capped)


def _draw_world_box(ax, world: WorldData) -> None:
    """Narysuj wireframe pudełka świata (12 krawędzi)."""
    xs = [world.min_bounds[0], world.max_bounds[0]]
    ys = [world.min_bounds[1], world.max_bounds[1]]
    zs = [world.min_bounds[2], world.max_bounds[2]]
    edges = [
        ([xs[0], xs[1]], [ys[0], ys[0]], [zs[0], zs[0]]),
        ([xs[0], xs[1]], [ys[1], ys[1]], [zs[0], zs[0]]),
        ([xs[0], xs[0]], [ys[0], ys[1]], [zs[0], zs[0]]),
        ([xs[1], xs[1]], [ys[0], ys[1]], [zs[0], zs[0]]),
        ([xs[0], xs[1]], [ys[0], ys[0]], [zs[1], zs[1]]),
        ([xs[0], xs[1]], [ys[1], ys[1]], [zs[1], zs[1]]),
        ([xs[0], xs[0]], [ys[0], ys[1]], [zs[1], zs[1]]),
        ([xs[1], xs[1]], [ys[0], ys[1]], [zs[1], zs[1]]),
        ([xs[0], xs[0]], [ys[0], ys[0]], [zs[0], zs[1]]),
        ([xs[1], xs[1]], [ys[0], ys[0]], [zs[0], zs[1]]),
        ([xs[0], xs[0]], [ys[1], ys[1]], [zs[0], zs[1]]),
        ([xs[1], xs[1]], [ys[1], ys[1]], [zs[0], zs[1]]),
    ]
    for x_pair, y_pair, z_pair in edges:
        ax.plot(x_pair, y_pair, z_pair, color="black", linewidth=0.7, linestyle="--", alpha=0.6)


def _draw_cylinder(ax, x: float, y: float, z: float, radius: float, height: float,
                   color: str = "#1b7a3f", n_seg: int = 24) -> None:
    """Narysuj cylindryczną przeszkodę jako siatkę powierzchni."""
    theta = np.linspace(0.0, 2.0 * np.pi, n_seg)
    th, zz = np.meshgrid(theta, np.linspace(z, z + height, 2))
    xx = x + radius * np.cos(th)
    yy = y + radius * np.sin(th)
    ax.plot_surface(xx, yy, zz, color=color, alpha=0.55, linewidth=0, antialiased=True)


def _draw_box(ax, x: float, y: float, z: float, length_x: float, width_y: float, height_z: float,
              color: str = "#1f5da4") -> None:
    """Narysuj prostopadłościenną przeszkodę jako 6 ścian (Poly3DCollection)."""
    lx, ly, lz = length_x / 2.0, width_y / 2.0, height_z
    x0, x1 = x - lx, x + lx
    y0, y1 = y - ly, y + ly
    z0, z1 = z, z + lz
    verts = [
        [(x0, y0, z0), (x1, y0, z0), (x1, y1, z0), (x0, y1, z0)],
        [(x0, y0, z1), (x1, y0, z1), (x1, y1, z1), (x0, y1, z1)],
        [(x0, y0, z0), (x1, y0, z0), (x1, y0, z1), (x0, y0, z1)],
        [(x0, y1, z0), (x1, y1, z0), (x1, y1, z1), (x0, y1, z1)],
        [(x0, y0, z0), (x0, y1, z0), (x0, y1, z1), (x0, y0, z1)],
        [(x1, y0, z0), (x1, y1, z0), (x1, y1, z1), (x1, y0, z1)],
    ]
    poly = Poly3DCollection(verts, facecolor=color, edgecolor="black", linewidths=0.4, alpha=0.55)
    ax.add_collection3d(poly)


def draw_world_3d(world: WorldData, obstacles: ObstaclesData,
                  start_positions: np.ndarray, target_positions: np.ndarray,
                  max_aspect: float = 4.0, title: str = "") -> None:
    """Renderuj widok 3D z proporcjonalną skalą osi i markerami start/cel."""
    fig = plt.figure(figsize=(13, 7))
    ax = fig.add_subplot(111, projection="3d")

    _draw_world_box(ax, world)
    for row in obstacles.data:
        if obstacles.shape_type == ObstacleShape.CYLINDER:
            x, y, z, radius, height, _ = row
            _draw_cylinder(ax, x, y, z, radius=radius, height=height)
        else:
            x, y, z, length_x, width_y, height_z = row
            _draw_box(ax, x, y, z, length_x=length_x, width_y=width_y, height_z=height_z)

    ax.scatter(start_positions[:, 0], start_positions[:, 1], start_positions[:, 2],
               c="#ffd54f", s=70, edgecolor="black", linewidth=0.8, label="start")
    ax.scatter(target_positions[:, 0], target_positions[:, 1], target_positions[:, 2],
               c="#ef5350", s=70, marker="*", edgecolor="black", linewidth=0.8, label="cel")

    natural_aspect = world.dimensions / np.min(world.dimensions)
    box_aspect = _compute_box_aspect(world, max_aspect=max_aspect)
    ax.set_box_aspect(box_aspect)

    ax.set_xlabel("X [m]  (szerokość)")
    ax.set_ylabel("Y [m]  (kierunek misji)")
    ax.set_zlabel("Z [m]")
    skala_info = ""
    if not np.allclose(natural_aspect, np.array(box_aspect) / np.min(box_aspect), atol=1e-3):
        skala_info = (f"\n(skalowanie osi: naturalna proporcja {natural_aspect.round(2).tolist()} "
                      f"→ display {[round(v, 2) for v in box_aspect]})")
    ax.set_title(f"{title}{skala_info}", fontsize=10)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
draw_world_3d(
    world=world,
    obstacles=obstacles,
    start_positions=start_positions,
    target_positions=target_positions,
    max_aspect=MAX_DISPLAY_ASPECT,
    title=(f"Świat '{env_cfg['name']}' — strategia {strategy_name}, "
           f"{obstacles.count} przeszkód typu {obstacles.shape_type.name}"),
)

## Notatka metodologiczna

Skalowanie osi wprowadzane przez `_compute_box_aspect` (limit
`MAX_DISPLAY_ASPECT`) zmienia *wyświetlanie*, ale nie wpływa na rzeczywiste
wymiary obiektów ani na działanie algorytmów. To czysto graficzny zabieg
umożliwiający percepcję wydłużonych korytarzy (forest 600 m / 60 m =
współczynnik 10, urban 1000 m / 300 m ≈ 3.3). W eksperymentach algorytmy
planowania operują na *prawdziwych* odległościach `WorldData.dimensions`.

Dla pełnego zachowania proporcji ustaw `MAX_DISPLAY_ASPECT = 10**6` —
wykres stanie się prawie liniowy dla forest, ale odległości pomiędzy
przeszkodami zachowują skalę 1:1:1.